In [1]:
import numpy as np

# =============================================================
# KOMPONEN 1: Decision Stump
# Pohon sangat sederhana — hanya 1 split (1 pertanyaan ya/tidak)
# Contoh: "apakah X < 25?" → jika ya prediksi A, jika tidak prediksi B
# =============================================================
class DecisionStump:
    def __init__(self):
        self.split_feature = None   # kolom mana yang dipakai split
        self.split_value   = None   # nilai threshold-nya
        self.left_value    = None   # prediksi jika X <= threshold
        self.right_value   = None   # prediksi jika X > threshold

    def fit(self, X, residuals):
        """Cari split terbaik yang meminimalkan MSE pada residual."""
        best_mse = float('inf')

        for feature in range(X.shape[1]):
            # Coba setiap nilai unik sebagai kandidat threshold
            thresholds = np.unique(X[:, feature])

            for threshold in thresholds:
                left_mask  = X[:, feature] <= threshold
                right_mask = ~left_mask

                # Butuh minimal 1 data di setiap sisi
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                # Prediksi optimal tiap sisi = rata-rata residual di sisi itu
                left_pred  = residuals[left_mask].mean()
                right_pred = residuals[right_mask].mean()

                # Hitung total MSE dari split ini
                left_mse  = ((residuals[left_mask]  - left_pred)  ** 2).sum()
                right_mse = ((residuals[right_mask] - right_pred) ** 2).sum()
                total_mse = left_mse + right_mse

                if total_mse < best_mse:
                    best_mse = total_mse
                    self.split_feature = feature
                    self.split_value   = threshold
                    self.left_value    = left_pred
                    self.right_value   = right_pred

    def predict(self, X):
        """Terapkan split yang sudah ditemukan ke data baru."""
        preds = np.where(
            X[:, self.split_feature] <= self.split_value,
            self.left_value,
            self.right_value
        )
        return preds


# =============================================================
# KOMPONEN 2: Gradient Boosting Regressor
# Membangun model secara bertahap — setiap pohon memperbaiki
# kesalahan pohon-pohon sebelumnya
# =============================================================
class GradientBoostingFromScratch:
    def __init__(self, n_estimators=3, learning_rate=0.5):
        self.n_estimators  = n_estimators   # jumlah pohon (iterasi)
        self.learning_rate = learning_rate  # seberapa "percaya" kita pada tiap pohon
        self.trees         = []             # daftar pohon yang sudah dilatih
        self.F0            = None           # prediksi awal (nilai rata-rata)

    def fit(self, X, y):
        # --- LANGKAH 0: Prediksi awal = rata-rata y ---
        # Tanpa info apapun, tebakan terbaik kita adalah rata-rata
        self.F0 = np.mean(y)
        F = np.full(len(y), self.F0)   # prediksi saat ini untuk semua data

        print(f"F0 (prediksi awal, rata-rata y) = {self.F0:.2f}")
        print(f"y asli                          = {y}\n")

        for m in range(1, self.n_estimators + 1):
            # --- LANGKAH 1: Hitung residual (kesalahan saat ini) ---
            # Untuk MSE loss, residual = negatif gradien = y - F(x)
            residuals = y - F
            print(f"--- Iterasi {m} ---")
            print(f"  Residual (y - prediksi) = {np.round(residuals, 2)}")

            # --- LANGKAH 2: Latih pohon baru untuk memprediksi residual ---
            tree = DecisionStump()
            tree.fit(X, residuals)
            tree_pred = tree.predict(X)
            print(f"  Prediksi pohon baru      = {np.round(tree_pred, 2)}")

            # --- LANGKAH 3: Perbarui prediksi dengan kontribusi pohon baru ---
            # Learning rate mengontrol "langkah" — kecil = hati-hati, besar = cepat
            F = F + self.learning_rate * tree_pred
            print(f"  Prediksi kumulatif       = {np.round(F, 2)}\n")

            self.trees.append(tree)

    def predict(self, X):
        # Mulai dari F0, lalu tambahkan kontribusi semua pohon
        F = np.full(X.shape[0], self.F0)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return F


# =============================================================
# EKSEKUSI: 5 baris data
# X = umur, y = berat badan (kg)
# =============================================================
X = np.array([[10], [20], [30], [40], [50]])
y = np.array([30.0, 50.0, 65.0, 75.0, 80.0])

model = GradientBoostingFromScratch(n_estimators=3, learning_rate=0.5)
model.fit(X, y)

print("=" * 45)
print(f"Prediksi akhir : {np.round(model.predict(X), 2)}")
print(f"Nilai asli     : {y}")
print(f"Error (MAE)    : {np.round(np.abs(model.predict(X) - y).mean(), 2)}")

F0 (prediksi awal, rata-rata y) = 60.00
y asli                          = [30. 50. 65. 75. 80.]

--- Iterasi 1 ---
  Residual (y - prediksi) = [-30. -10.   5.  15.  20.]
  Prediksi pohon baru      = [-20.   -20.    13.33  13.33  13.33]
  Prediksi kumulatif       = [50.   50.   66.67 66.67 66.67]

--- Iterasi 2 ---
  Residual (y - prediksi) = [-20.     0.    -1.67   8.33  13.33]
  Prediksi pohon baru      = [-20.   5.   5.   5.   5.]
  Prediksi kumulatif       = [40.   52.5  69.17 69.17 69.17]

--- Iterasi 3 ---
  Residual (y - prediksi) = [-10.    -2.5   -4.17   5.83  10.83]
  Prediksi pohon baru      = [-5.56 -5.56 -5.56  8.33  8.33]
  Prediksi kumulatif       = [37.22 49.72 66.39 73.33 73.33]

Prediksi akhir : [37.22 49.72 66.39 73.33 73.33]
Nilai asli     : [30. 50. 65. 75. 80.]
Error (MAE)    : 3.44
